Facebook data downloaded from Humanitarian Data Exchange on 17th January 2025.
Links for 2020 and 2021-2022 data are: https://data.humdata.org/dataset/c3429f0e-651b-4788-bb2f-4adbf222c90e/resource/55a51014-0d27-49ae-bf92-c82a570c2c6c/download/movement-range-data-2022-05-22.zip
and https://data.humdata.org/dataset/c3429f0e-651b-4788-bb2f-4adbf222c90e/resource/3d77ce5c-ab6d-4864-b8a2-c8bafffac4f3/download/movement-range-data-2020-03-01-2020-12-31.zip
which are accessible from:
https://data.humdata.org/dataset/movement-range-maps?

In [1]:
import pandas as pd
import pycountry
import re
from emu_renewal.inputs import RAW_MOB_PATH, DATA_PATH, extract_country_from_fb_mobility, extract_country_from_euro_pop, replace_chars_for_map, get_weighted_average_from_df

In [2]:
# Get all Facebook mobility data
req_cols = ["ds", "country", "polygon_name", "all_day_bing_tiles_visited_relative_change", "all_day_ratio_single_tile_users"]
data20 = pd.read_csv(RAW_MOB_PATH / "movement-range-data-2020-03-01--2020-12-31.txt", sep="\t", index_col="ds", usecols=req_cols)
data21_22 = pd.read_csv(RAW_MOB_PATH / "movement-range-2022-05-22.txt", sep="\t", index_col="ds", usecols=req_cols)
fb_data = pd.concat([data20, data21_22])
fb_data.index = pd.to_datetime(fb_data.index)

In [3]:
# Get all European population data for 2020
nuts2_raw = pd.read_csv(DATA_PATH / "population/estat_demo_r_pjangroup_filtered_en.csv.gz", index_col="geo", usecols=["geo", "OBS_VALUE", "TIME_PERIOD"])
nuts2_pops = nuts2_raw.loc[nuts2_raw["TIME_PERIOD"] == 2020, "OBS_VALUE"]

In [4]:
country = "Spain"

In [5]:
country_mobility = extract_country_from_fb_mobility(fb_data, country)
country_pop_map = extract_country_from_euro_pop(nuts2_pops, country)
country_pop_map.index = replace_chars_for_map(country_pop_map.index)
country_pop_map = country_pop_map[~country_pop_map.index.duplicated(keep='first')]
if country == "Spain":
    country_pop_map["Ceuta y Melilla"] = country_pop_map["Ciudad de Ceuta"] + country_pop_map["Ciudad de Melilla"]
country_mobility.loc[:, "polygon_name"] = replace_chars_for_map(country_mobility["polygon_name"])
country_mobility.loc[:, ["pop"]] = country_mobility["polygon_name"].map(country_pop_map)
final_mobility_data = get_weighted_average_from_df(country_mobility, "all_day_bing_tiles_visited_relative_change", "pop")

In [6]:
final_mobility_data.to_csv(DATA_PATH / f"mobility/{pycountry.countries.get(name=country).alpha_2}_fbmob_data.csv")

In [7]:
# Regions failed to map
set([i for i in country_mobility["polygon_name" ] if i not in country_pop_map.index])

{'Islas Baleares', 'Islas Canarias'}